# Prepare conection by first time

## Connecting Google Colab to GitHub using SSH (Complete Setup Guide)

This guide explains how to connect a Google Colab project stored in Google Drive to GitHub using SSH authentication. The goal is to work directly from Colab and push updates to GitHub without using passwords or tokens.

---

# Recommended Project Structure

```text
Google Drive
│
├── lmf/                <- Git repository
├── data/               <- Large datasets (NOT tracked by GitHub)
├── output/             <- Large outputs/results (NOT tracked by GitHub)
└── docs/
    └── keys/           <- SSH keys (OUTSIDE the repository)
```

Where:

- `lmf/` contains notebooks, scripts, and versioned files
- `data/` and `output/` remain outside GitHub
- `keys/` stores SSH keys permanently
- SSH keys MUST remain outside the Git repository for security reasons

---

# IMPORTANT SECURITY RECOMMENDATION

The SSH private key:

```text
id_ed25519
```

MUST NEVER be stored inside the Git repository folder.

Correct:

```text
/content/drive/MyDrive/docs/keys/id_ed25519
```

Incorrect:

```text
/content/drive/MyDrive/lmf/id_ed25519
```

If the key is stored inside the repository, it could accidentally be uploaded to GitHub and compromise your account.

Always keep SSH keys in an external directory such as:

```text
/content/drive/MyDrive/docs/keys/
```

The repository should NEVER contain:

- SSH keys
- API keys
- passwords
- `.env` files
- credentials
- tokens

---

# STEP 1 — Create a GitHub Repository

In GitHub:

1. Go to:

```text
https://github.com/new
```

2. Create a new repository.

Example:

```text
lmf
```

3. The repository can be:
- public
- private

4. Do NOT initialize with:
- README
- `.gitignore`
- license

---

# STEP 2 — Open Google Colab and Mount Google Drive

```python
from google.colab import drive
drive.mount('/content/drive')
```

---

# STEP 3 — Move into Your Project Folder

Example:

```python
%cd /content/drive/MyDrive/lmf
```

If the folder does not exist:

```bash
!mkdir -p /content/drive/MyDrive/lmf
```

---

# STEP 4 — Initialize a Git Repository

Inside the project folder:

```bash
!git init
```

---

# STEP 5 — Configure Git Username and Email

Set your Git identity:

```bash
!git config --global user.name "Your Name"
!git config --global user.email "your_email@example.com"
```

Verify configuration:

```bash
!git config --list
```

---

# STEP 6 — Generate SSH Keys

Create a folder to store SSH keys:

```bash
!mkdir -p /content/drive/MyDrive/docs/keys
```

Generate the SSH key:

```bash
!ssh-keygen -t ed25519 -C "your_email@example.com" -f /content/drive/MyDrive/docs/keys/id_ed25519 -N ""
```

This creates:

```text
id_ed25519
id_ed25519.pub
```

Where:

- `id_ed25519` = private key
- `id_ed25519.pub` = public key

---

# STEP 7 — Copy the Public SSH Key

Display the public key:

```bash
!cat /content/drive/MyDrive/docs/keys/id_ed25519.pub
```

Copy the entire output.

It should look similar to:

```text
ssh-ed25519 AAAAC3Nza... your_email@example.com
```

---

# STEP 8 — Add SSH Key to GitHub

In GitHub:

```text
Settings
→ SSH and GPG keys
→ New SSH key
```

Fill in:

## Title

```text
Colab
```

## Key

Paste the copied public SSH key.

Save.

---

# STEP 9 — Load SSH Key into the Current Colab Session

IMPORTANT:

This step must be executed every time a new Colab runtime starts.

```bash
!mkdir -p ~/.ssh

!cp /content/drive/MyDrive/docs/keys/id_ed25519 ~/.ssh/id_ed25519
!cp /content/drive/MyDrive/docs/keys/id_ed25519.pub ~/.ssh/id_ed25519.pub

!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519
!chmod 644 ~/.ssh/id_ed25519.pub

!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts
```

---

# STEP 10 — Verify SSH Connection

```bash
!ssh -T git@github.com
```

You should see something like:

```text
Hi username! You've successfully authenticated...
```

---

# STEP 11 — Connect the Local Repository to GitHub

Add the GitHub SSH remote:

```bash
!git remote add origin git@github.com:USERNAME/REPOSITORY.git
```

If the remote already exists:

```bash
!git remote set-url origin git@github.com:USERNAME/REPOSITORY.git
```

Verify:

```bash
!git remote -v
```

Expected output:

```text
origin  git@github.com:USERNAME/REPOSITORY.git (fetch)
origin  git@github.com:USERNAME/REPOSITORY.git (push)
```

---

# STEP 12 — Create a `.gitignore` File

Create `.gitignore`:

```bash
%%writefile .gitignore
.ipynb_checkpoints/
__pycache__/
*.pyc

# environments
.env
.venv/

# datasets
data/
output/

# bioinformatics
*.bam
*.fastq
*.fq
*.h5
*.hdf5

# images
*.tiff

# security
*.pem
*.key
id_ed25519
id_ed25519.pub
.ssh/
keys/
```

---

# STEP 13 — First Commit

Add files:

```bash
!git add .
```

Create commit:

```bash
!git commit -m "Initial commit"
```

Create main branch:

```bash
!git branch -M main
```

Push to GitHub:

```bash
!git push -u origin main
```

This only needs to be done once.

---

# Daily Workflow

Every time you open a new Colab runtime:

---

## 1. Mount Google Drive

```python
from google.colab import drive
drive.mount('/content/drive')
```

---

## 2. Move into the Repository

```python
%cd /content/drive/MyDrive/lmf
```

---

## 3. Load SSH Key

```bash
!mkdir -p ~/.ssh

!cp /content/drive/MyDrive/docs/keys/id_ed25519 ~/.ssh/id_ed25519
!cp /content/drive/MyDrive/docs/keys/id_ed25519.pub ~/.ssh/id_ed25519.pub

!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!ssh-keyscan github.com >> ~/.ssh/known_hosts
```

---

## 4. Push Changes

```bash
!git add .
!git commit -m "Update notebooks"
!git push
```

---

# Optional Automatic Push Function

```python
import subprocess

def git_push(msg="update"):

    status = subprocess.run(
        ["git", "status", "--porcelain"],
        capture_output=True,
        text=True
    )

    if not status.stdout.strip():
        print("No changes to push.")
        return

    commands = [
        ["git", "add", "."],
        ["git", "commit", "-m", msg],
        ["git", "push"]
    ]

    for cmd in commands:
        result = subprocess.run(cmd, capture_output=True, text=True)

        print(f"\n>>> {' '.join(cmd)}")
        print(result.stdout)

        if result.stderr:
            print(result.stderr)
```

Usage:

```python
git_push("new metabolomics analysis")
```

---

# Important Recommendations

## Files that SHOULD go to GitHub

- notebooks
- scripts
- README files
- configuration files
- lightweight figures
- documentation

## Files that SHOULD NOT go to GitHub

- raw datasets
- large TIFF images
- FASTQ/BAM files
- large outputs
- temporary files
- credentials
- SSH keys
- API keys
- tokens

---

# Useful Git Commands

Check repository status:

```bash
!git status
```

Check configured remotes:

```bash
!git remote -v
```

Pull latest changes:

```bash
!git pull
```

Push changes:

```bash
!git push
```

View commit history:

```bash
!git log --oneline
```

---

# Daily workflow to commit on Github

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/lmf

In [17]:
!mkdir -p ~/.ssh

!cp /content/drive/MyDrive/docs/keys/id_ed25519 ~/.ssh/id_ed25519
!cp /content/drive/MyDrive/docs/keys/id_ed25519.pub ~/.ssh/id_ed25519.pub

!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519
!chmod 644 ~/.ssh/id_ed25519.pub

!ssh-keyscan github.com >> ~/.ssh/known_hosts

# github.com:22 SSH-2.0-47e7538
# github.com:22 SSH-2.0-47e7538
# github.com:22 SSH-2.0-47e7538
# github.com:22 SSH-2.0-47e7538
# github.com:22 SSH-2.0-47e7538


In [23]:
!ssh -T git@github.com

Hi maurope! You've successfully authenticated, but GitHub does not provide shell access.


In [19]:
!git status

Refresh index: 100% (159/159), done.
On branch main
nothing to commit, working tree clean


In [20]:
!git add .
!git commit -m "Update notebooks"
#!git push -u origin main       # Only use the first time
!git push

[main 8d15b3b] Update notebooks
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 198, done.
Counting objects: 100% (198/198), done.
Delta compression using up to 2 threads
Compressing objects: 100% (198/198), done.
Writing objects: 100% (198/198), 44.27 MiB | 3.13 MiB/s, done.
Total 198 (delta 48), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (48/48), done.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File data/04_15_2026_sql_data_frame_creation/subset_2/6-Gas Production-Subset 2_(2025)_CIAT 02032026 IsaMo.xlsx is 94.58 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: File output/2026_05_07_methane_paper/figure_2.tiff is 57.38 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To github.com:maurope/lmf.git
 * [new branch]      main 

In [21]:
!git status

Refresh index: 100% (159/159), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   02_github.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
